In [ ]:
# importing related libraried
import json
from os import path
import pandas as pd
import openai


#Set root path to the location of open oai_finetune_modules
import os
os.chdir("/llm_coding/")
import oai_finetune_modules as oaimd

In [ ]:

# 0. SETTING PATHS AND CONSTANTS
cv_param = "cv1" # Choice of cv1, cv2, cv3, cv4 and cv5
validation_rootpath = "/cv_dfs/"
neg_fpath = "/neg_hiv_msgs.csv"
control_fpath = "/control_msgs.txt"

op_rootpath = "/validation_outputs"
with open("cv_recode_validation_params.json") as f:
    validation_params = json.load(f)
validation_params = validation_params[cv_param]

validation_fpath = path.join(validation_rootpath, validation_params["valid_fname"])
health_gpt_modelid = validation_params["health_gpt_id"]
quant_gpt_modelid = validation_params["quant_gpt_id"]
tgt_gpt_modelid = validation_params["tgt_gpt_id"]

validation_df = pd.read_csv(validation_fpath)
neg_df = pd.read_csv(neg_fpath)
control_msgs = []
with open(control_fpath, "r") as ip_fp:
    control_msgs = ip_fp.readlines()


annotation_type = "rec_rating"

qs_to_validate = ["act_2", "act_3", "act_4", "act_5", "eff_1", "eff_2", "eff_3", "eff_4"]

reqd_val_cols = ["Content"] + [f"{annotation_type}_{q}" for q in qs_to_validate]
reqd_validation_df = validation_df[reqd_val_cols]
annotation_cols = [colname for colname in reqd_validation_df if annotation_type in colname]
reqd_validation_df = reqd_validation_df.melt(id_vars="Content", value_vars = annotation_cols)
reqd_validation_df["qcode"] = reqd_validation_df["variable"].str.replace(f"{annotation_type}_", "")
reqd_validation_df["annotation_type"] = reqd_validation_df["variable"].str.replace("_[0-9]", "", regex=True)

annotation_combs = [f"{annotation_type}_{q}" for q in qs_to_validate]

control_df_comb = [pd.DataFrame({"Content": control_msgs, "annotation_comb": annotation}) for annotation in annotation_combs]
neg_twt_df_comb = [pd.DataFrame({"Content": neg_df["Content"], "annotation_comb": annotation}) for annotation in annotation_combs]

control_df_comb = pd.concat(control_df_comb).reset_index(drop=True)
neg_twt_df_comb = pd.concat(neg_twt_df_comb).reset_index(drop=True)

control_df_comb["annotation_type"] = control_df_comb["annotation_comb"].str.replace("_[0-9]", "", regex=True)
neg_twt_df_comb["annotation_type"] = neg_twt_df_comb["annotation_comb"].str.replace("_[0-9]", "", regex=True)
control_df_comb["qcode"] = control_df_comb["annotation_comb"].str.replace(f"{annotation_type}_", "")
neg_twt_df_comb["qcode"] = neg_twt_df_comb["annotation_comb"].str.replace(f"{annotation_type}_", "")

In [ ]:
# 1. CREATING VALIDATION PROMPTS
## 1.1. Generating the prompt dictionaries from the validation file
qnt_prompts = [[{"role": "system", "content": f"{oaimd.ID_STRS['qnt']} {oaimd.ANNOTATION_TYPE_PROMPTS[atype]} {oaimd.Q_PROMPTS[qcode]}"},
                {"role": "user", "content": f"Rate the following tweet \"{row_content}\""}]
               for atype, qcode, row_content 
               in zip(reqd_validation_df["annotation_type"], reqd_validation_df["qcode"], reqd_validation_df["Content"])]
hlt_prompts = [[{"role": "system", "content": f"{oaimd.ID_STRS['hlt']} {oaimd.ANNOTATION_TYPE_PROMPTS[atype]} {oaimd.Q_PROMPTS[qcode]}"},
                {"role": "user", "content": f"Rate the following tweet \"{row_content}\""}]
               for atype, qcode, row_content 
               in zip(reqd_validation_df["annotation_type"], reqd_validation_df["qcode"], reqd_validation_df["Content"])]
tgt_prompts = [[{"role": "system", "content": f"{oaimd.ID_STRS['tgt']} {oaimd.ANNOTATION_TYPE_PROMPTS[atype]} {oaimd.Q_PROMPTS[qcode]}"},
                {"role": "user", "content": f"Rate the following tweet \"{row_content}\""}]
               for atype, qcode, row_content 
               in zip(reqd_validation_df["annotation_type"], reqd_validation_df["qcode"], reqd_validation_df["Content"])]

## 1.2. Generating the prompt dictionaries from the negative and control tweet file
neg_qnt_prompts = [[{"role": "system", "content": f"{oaimd.ID_STRS['qnt']} {oaimd.ANNOTATION_TYPE_PROMPTS[atype]} {oaimd.Q_PROMPTS[qcode]}"},
                   {"role": "user", "content": f"Rate the following tweet \"{row_content}\""}]
                  for atype, qcode, row_content 
                  in zip(neg_twt_df_comb["annotation_type"], neg_twt_df_comb["qcode"], neg_twt_df_comb["Content"])]
neg_hlt_prompts = [[{"role": "system", "content": f"{oaimd.ID_STRS['hlt']} {oaimd.ANNOTATION_TYPE_PROMPTS[atype]} {oaimd.Q_PROMPTS[qcode]}"},
                   {"role": "user", "content": f"Rate the following tweet \"{row_content}\""}]
                  for atype, qcode, row_content 
                  in zip(neg_twt_df_comb["annotation_type"], neg_twt_df_comb["qcode"], neg_twt_df_comb["Content"])]
neg_tgt_prompts = [[{"role": "system", "content": f"{oaimd.ID_STRS['tgt']} {oaimd.ANNOTATION_TYPE_PROMPTS[atype]} {oaimd.Q_PROMPTS[qcode]}"},
                   {"role": "user", "content": f"Rate the following tweet \"{row_content}\""}]
                  for atype, qcode, row_content 
                  in zip(neg_twt_df_comb["annotation_type"], neg_twt_df_comb["qcode"], neg_twt_df_comb["Content"])]
ctrl_qnt_prompts = [[{"role": "system", "content": f"{oaimd.ID_STRS['qnt']} {oaimd.ANNOTATION_TYPE_PROMPTS[atype]} {oaimd.Q_PROMPTS[qcode]}"},
                    {"role": "user", "content": f"Rate the following tweet \"{row_content}\""}]
                   for atype, qcode, row_content 
                   in zip(control_df_comb["annotation_type"], control_df_comb["qcode"], control_df_comb["Content"])]
ctrl_hlt_prompts = [[{"role": "system", "content": f"{oaimd.ID_STRS['hlt']} {oaimd.ANNOTATION_TYPE_PROMPTS[atype]} {oaimd.Q_PROMPTS[qcode]}"},
                    {"role": "user", "content": f"Rate the following tweet \"{row_content}\""}]
                   for atype, qcode, row_content 
                   in zip(control_df_comb["annotation_type"], control_df_comb["qcode"], control_df_comb["Content"])]
ctrl_tgt_prompts = [[{"role": "system", "content": f"{oaimd.ID_STRS['tgt']} {oaimd.ANNOTATION_TYPE_PROMPTS[atype]} {oaimd.Q_PROMPTS[qcode]}"},
                    {"role": "user", "content": f"Rate the following tweet \"{row_content}\""}]
                   for atype, qcode, row_content 
                   in zip(control_df_comb["annotation_type"], control_df_comb["qcode"], control_df_comb["Content"])]

In [ ]:

# 2. GETTING GPT'S RESPONSES
client = openai.OpenAI(api_key=oaimd.API_KEY)

hlth_tune_responses = [client.chat.completions.create(
                        model=health_gpt_modelid,
                        messages=prompt).choices[0].message.content
                       for prompt in hlt_prompts]
tgt_tune_responses = [client.chat.completions.create(
                       model=tgt_gpt_modelid,
                       messages=prompt).choices[0].message.content
                      for prompt in tgt_prompts]
qnt_tune_responses = [client.chat.completions.create(
                        model=quant_gpt_modelid,
                        messages=prompt).choices[0].message.content
                       for prompt in qnt_prompts]

neg_hlth_tune_responses = [client.chat.completions.create(
                            model=health_gpt_modelid,
                            messages=prompt).choices[0].message.content
                           for prompt in neg_hlt_prompts]
neg_tgt_tune_responses = [client.chat.completions.create(
                           model=tgt_gpt_modelid,
                           messages=prompt).choices[0].message.content
                          for prompt in neg_tgt_prompts]
neg_qnt_tune_responses = [client.chat.completions.create(
                            model=quant_gpt_modelid,
                            messages=prompt).choices[0].message.content
                           for prompt in neg_qnt_prompts]

ctrl_hlth_tune_responses = [client.chat.completions.create(
                             model=health_gpt_modelid,
                             messages=prompt).choices[0].message.content
                            for prompt in ctrl_hlt_prompts]
ctrl_tgt_tune_responses = [client.chat.completions.create(
                            model=tgt_gpt_modelid,
                            messages=prompt).choices[0].message.content
                           for prompt in ctrl_tgt_prompts]
ctrl_qnt_tune_responses = [client.chat.completions.create(
                             model=quant_gpt_modelid,
                             messages=prompt).choices[0].message.content
                            for prompt in ctrl_qnt_prompts]

In [ ]:

# 3. GENERATING OUTPUT FILE
op_file = reqd_validation_df
op_file["health_gpt"] = pd.Series(hlth_tune_responses)
op_file["tgt_group_gpt"] = pd.Series(tgt_tune_responses)
op_file["quant_gpt"] = pd.Series(qnt_tune_responses)

neg_op_file = neg_twt_df_comb
neg_op_file["health_gpt"] = pd.Series(neg_hlth_tune_responses)
neg_op_file["tgt_group_gpt"] = pd.Series(neg_tgt_tune_responses)
neg_op_file["quant_gpt"] = pd.Series(neg_qnt_tune_responses)

ctrl_op_file = control_df_comb
ctrl_op_file["health_gpt"] = pd.Series(ctrl_hlth_tune_responses)
ctrl_op_file["tgt_group_gpt"] = pd.Series(ctrl_tgt_tune_responses)
ctrl_op_file["quant_gpt"] = pd.Series(ctrl_qnt_tune_responses)


op_fpath = path.join(op_rootpath, f"{cv_param}_{annotation_type}_apr22_fine_tune_validation.csv")
neg_op_fpath = path.join(op_rootpath, f"{cv_param}_{annotation_type}_apr22_fine_tune_validation_neg_twts.csv")
ctrl_op_fpath = path.join(op_rootpath, f"{cv_param}_{annotation_type}_apr22_fine_tune_validation_ctrl_twts.csv")
op_file.to_csv(op_fpath, index=False)
neg_op_file.to_csv(neg_op_fpath, index=False)
ctrl_op_file.to_csv(ctrl_op_fpath, index=False)